# Titanic Survival Prediction
This notebook demonstrates a complete machine learning workflow to predict passenger survival on the Titanic using Logistic Regression.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.linear_model import LogisticRegression

## 1. Data Loading and Initial Exploration
First, we load the dataset and take a look at a few sample rows to understand the features we are working with.

In [2]:
df = pd.read_csv("titanic_data_updated.csv")

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
409,410,no,third,"Lefebre, Miss. Ida",female,NaN,3,1,4133,25.4667,NaN,S
837,838,no,third,"Sirota, Mr. Maurice",male,NaN,0,0,392092,8.0500,NaN,S
33,34,no,second,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S
195,196,yes,first,"Lurette, Miss. Elise",female,58.0,0,0,PC 17569,146.5208,B80,C
287,288,no,third,"Naidenoff, Mr. Penko",male,22.0,0,0,349206,7.8958,NaN,S


In [3]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


## 2. Feature Engineering
We create new features like `Family_Size` and extract the `Deck` from the `Cabin` column to provide more meaningful information to the model.

In [6]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]

df.sample(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
585,586,yes,first,"Taussig, Miss. Ruth",female,18.0,0,2,110413,79.6500,E68,S,3,E
711,712,no,first,"Klaber, Mr. Herman",male,NaN,0,0,113028,26.5500,C124,S,1,C
301,302,yes,third,"McCoy, Mr. Bernard",male,NaN,2,0,367226,23.2500,Missing,Q,3,M
826,827,no,third,"Lam, Mr. Len",male,NaN,0,0,1601,56.4958,Missing,S,1,M
858,859,yes,third,"Baclini, Mrs. Solomon (Latifa Qurban)",female,24.0,0,3,2666,19.2583,Missing,C,4,M
389,390,yes,second,"Lehmann, Miss. Bertha",female,17.0,0,0,SC 1748,12.0000,Missing,C,1,M


In [8]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [9]:
X = df.drop('Survived', axis=1)
y = df['Survived']

## 3. Data Splitting
We split the data into training (to teach the model) and testing (to evaluate the model) sets. This ensures we can test the model on data it hasn't seen before.

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E
